# Structural Design Patterns

These 3 patterns all answer one question:

> **"How do I combine or wrap objects cleanly, without creating a mess?"**

All three work by **wrapping** one object inside another object. That is exactly why people mix them up.
The trick to telling them apart is to ask **WHY** the wrapping is happening:

| Pattern | Why you wrap the object | One-line definition |
|---|---|---|
| **Decorator** | to **ADD** behavior | Stack new behavior onto an object at runtime, without a subclass for every possible combination of features. |
| **Adapter** | to **TRANSLATE** an interface | Make an incompatible, pre-existing interface work with your code, without modifying the original class. |
| **Facade** | to **SIMPLIFY** a process | Hide a complicated, multi-step subsystem behind ONE simple method that does the whole job correctly. |

### Everyday analogies (read these first, they make the rest easy)

- **Decorator = gift wrapping.** The gift inside never changes. Each layer of wrapping adds something (paper, then a bow, then a card). You choose how many layers, and in what order, at the moment you wrap it.
- **Adapter = a travel plug adapter.** Your US laptop plug does not fit a UK wall socket. You do not rebuild the laptop and you do not rebuild the wall. You put a small adapter in between that translates one shape into the other.
- **Facade = the "Movie mode" button on a smart remote.** One press turns on the TV, switches the input, dims the lights and starts the soundbar. One simple button hides many fiddly steps.

---
# Part 1: Decorator

## 1.1 What problem are we solving?

**Goal:** add extra behavior to an object at **RUNTIME**, without writing a subclass for every possible combination of features.

> **"Runtime"** means *while the program is running*, based on real data (which user, which order, which setting).
> The opposite is deciding *while you are writing the code*, which can never change unless you edit the code.

### The example: a stock trading app

The core job is simple: **execute an order** (send a buy or sell to the stock market).

But sometimes you want extra things to happen around that core job:

- **Logging**: write down what happened, so you can investigate later.
- **Risk check**: validate the order before it goes to the market (for example, "is this order too big?").
- Sometimes **both**, sometimes **neither**.

And you want to decide **which ones, per order, at runtime**. For example: big orders get a risk check, small ones don't; logging is switched on only in debug mode.

If you hardcode every combination, your `Order` class keeps growing new responsibilities it was never meant to have. An order should know how to *execute*. It should not also be the logging system and the risk department.

## 1.2 A tiny bit of Python you need first: abstract base classes

An **interface** is a promise: *"any object of this type will have these methods."*
Python writes an interface using an **abstract base class (ABC)**:

- `from abc import ABC, abstractmethod` imports the tools.
- A class that inherits from `ABC` is a "template" class.
- A method marked `@abstractmethod` has **no real body**. Every child class **must** write its own version.
- You **cannot create an object** directly from a class that still has an unfinished abstract method. Python stops you with a `TypeError`. This is a safety net: it is impossible to forget to implement `execute()`.

Run the cell below to see the safety net in action.

In [1]:
from abc import ABC, abstractmethod


class Order(ABC):
    @abstractmethod
    def execute(self) -> None:
        """Every kind of Order MUST know how to execute itself."""
        ...


try:
    Order()  # not allowed: Order is only a template
except TypeError as error:
    print("Python stopped us:", error)

Python stopped us: Can't instantiate abstract class Order without an implementation for abstract method 'execute'


## 1.3 The naive approach (what most people try first)

"I need logging? I'll make a `LoggedOrder` subclass. I need a risk check? `RiskCheckedOrder`. I need both? `LoggedRiskCheckedOrder`."

In other words: **one subclass for every COMBINATION of features.**

Look closely at the code below. Notice how the line `print("Executing order")` (the real work) is **copied into every class**, and how the logging lines and risk-check lines are copied too.

In [2]:
# Naive approach: a subclass for every COMBINATION of features


class PlainOrder(Order):
    def execute(self):
        print("Executing order")


class LoggedOrder(Order):
    def execute(self):
        print("[LOG] Before execution")
        print("Executing order")            # copied
        print("[LOG] After execution")


class RiskCheckedOrder(Order):
    def execute(self):
        print("[RISK CHECK] Validating order")
        print("Executing order")            # copied again


class LoggedRiskCheckedOrder(Order):
    def execute(self):
        print("[LOG] Before execution")      # copied from LoggedOrder
        print("[RISK CHECK] Validating order")  # copied from RiskCheckedOrder
        print("Executing order")            # copied a third time
        print("[LOG] After execution")


LoggedRiskCheckedOrder().execute()

[LOG] Before execution
[RISK CHECK] Validating order
Executing order
[LOG] After execution


It works, so what is wrong with it? Three separate things.

### Problem 1: combinatorial explosion (the number of classes doubles with every feature)

Think of each feature as a **light switch** that is either ON or OFF.

- 1 switch has 2 positions: off, on.
- 2 switches have 2 x 2 = 4 combinations.
- 3 switches have 2 x 2 x 2 = 8 combinations.
- `n` switches have **2 to the power n** (written `2^n`) combinations.

With the naive approach, every combination needs its own class:

| Number of features | Classes needed (naive) | Classes needed (Decorator) |
|---|---|---|
| 1 (logging) | 2 | 1 base + 1 |
| 2 (+ risk check) | 4 | 1 base + 2 |
| 3 (+ rate limiting) | 8 | 1 base + 3 |
| 4 (+ notifications) | **16** | 1 base + 4 |
| 10 | **1,024** | 1 base + 10 |

With 4 independent features (logging, risk-check, rate-limiting, notifications) you'd need up to **16 subclasses** just to cover every combination. This is called **COMBINATORIAL EXPLOSION**. The naive approach grows as `2^n`; the Decorator approach grows as `n`.

### Problem 2: copy-paste everywhere

The real execution code lives in 4 places. If the way orders are sent to the market changes, you must find and fix all 4 (or 16, or 1,024). Miss one and you have a bug.

### Problem 3: you still can't decide at RUNTIME

Even with all 16 classes pre-written, the combination is **baked into which class name you typed** when writing the code. To choose at runtime you would need an ugly `if/elif` ladder like this:

```python
if needs_logging and needs_risk_check:
    order = LoggedRiskCheckedOrder()
elif needs_logging:
    order = LoggedOrder()
elif needs_risk_check:
    order = RiskCheckedOrder()
else:
    order = PlainOrder()
# ...and this ladder has 16 branches once you have 4 features.
```

## 1.4 The fix: wrap the order in LAYERS, one feature per layer, decided at runtime

Instead of one big class per combination, we build **small layers** and stack them, like gift wrapping:

```
+--------------------------------------------------+
|  LoggingDecorator        (wraps RiskCheckDecorator)  <- outermost layer
|  +--------------------------------------------+  |
|  |  RiskCheckDecorator    (wraps BasicOrder)   |  |  <- middle layer
|  |  +--------------------------------------+  |  |
|  |  |  BasicOrder  (the real order)         |  |  |  <- the gift in the middle
|  |  +--------------------------------------+  |  |
|  +--------------------------------------------+  |
+--------------------------------------------------+
```

The slide draws the same thing as a chain, where each arrow means "wraps":

```
BasicOrder
    ^  wraps
RiskCheckDecorator
    ^  wraps
LoggingDecorator
```

**Calling `execute()` on the OUTERMOST layer runs each layer's own code, then delegates inward, like unwrapping a gift.**

("Delegate" just means *"hand the job on to someone else"*. Each layer does its own little bit, then says to the layer inside it: "your turn".)

Three rules make this work:

1. **Every layer has the same shape.** `BasicOrder`, `RiskCheckDecorator` and `LoggingDecorator` are all `Order`s, and all have an `execute()` method. Whoever holds the outermost layer cannot tell (and does not care) whether it is a plain order or one wrapped 10 times.
2. **Every layer holds the thing inside it.** A decorator keeps a reference to the object it wraps.
3. **Every layer does its bit, then calls `execute()` on the thing inside.**

## 1.5 The same thing as a UML class diagram

UML is just a standard way of drawing classes as boxes. Each box has the class name on top, then its data, then its methods.

```
                    <<interface>>
                        Order
                  -----------------
                  + execute()
                   /             \
                  /               \     (both of these ARE Orders)
                 v                 v
        BasicOrder              OrderDecorator
      --------------       ----------------------------
      + execute()          # wrapped_order: Order        <-- it also HAS an Order inside
                           + execute()
                             /                  \
                            v                    v
                 LoggingDecorator          RiskCheckDecorator
                 ----------------          ------------------
                 + execute()               + execute()
```

How to read the symbols:

- `<<interface>>` means "this is a promise/template, not a real thing". In Python: an `ABC`.
- `+` means **public**: anyone may call it.
- `#` means **protected**: meant for this class and its children only. Python has no real "protected", so by convention we start the name with one underscore: `self._wrapped_order`.
- The arrows mean "inherits from / implements". (Real UML draws the "implements" arrow pointing UP from the child to the interface. The slide draws it downward just to make it easier to read.)

### The single most important thing in this diagram

`OrderDecorator` has **two** relationships with `Order` at the same time:

- It **IS an** `Order` (it inherits from `Order`), so it can be used anywhere an `Order` is expected.
- It **HAS an** `Order` (it stores one in `wrapped_order`), so it has something to pass the work on to.

"Is one AND has one" is the fingerprint of the Decorator pattern.

**Notice:** the Decorator and the thing it wraps (the "Component") share the **SAME interface** (`Order`). That is the signal that tells you it's **Decorator, not Adapter**. An Adapter always involves **TWO different interfaces** (you'll see this in Part 2).

## 1.6 The code, step by step

### Step 1: the interface (`Order`)

We already wrote it in section 1.2. Here it is again so this section reads on its own. It says: *"Every order must have an `execute()` method."*

In [3]:
from abc import ABC, abstractmethod


class Order(ABC):
    @abstractmethod
    def execute(self) -> None:
        """Every kind of Order MUST know how to execute itself."""
        ...

### Step 2: the real thing (`BasicOrder`)

This is the **gift in the middle**. It does the actual work: sending the order to the market. It knows **nothing** about logging or risk checks, and it never will. That is good: one class, one job.

In [4]:
class BasicOrder(Order):
    def execute(self) -> None:
        # In a real app this would send the order to the stock exchange.
        print("Executing order")

### Step 3: the base decorator (`OrderDecorator`)

This is the **common parent for every layer of wrapping paper**. It does two things:

1. Inherits from `Order` -> so a decorator **is an** `Order`.
2. Takes an `Order` in its constructor (`__init__`) and stores it as `self._wrapped_order` -> so a decorator **has an** `Order` inside it.

Its `execute()` simply passes the job straight through to the wrapped order. On its own it adds nothing; its children will add the interesting behavior.

Notice the type hint: `wrapped_order: Order`, **not** `BasicOrder`. That is what allows a decorator to wrap *another decorator*. A `RiskCheckDecorator` is an `Order`, so a `LoggingDecorator` is allowed to wrap it. This one detail is what makes stacking possible.

In [5]:
class OrderDecorator(Order):
    def __init__(self, wrapped_order: Order) -> None:
        # The thing being wrapped. It can be a BasicOrder OR another decorator.
        self._wrapped_order = wrapped_order

    def execute(self) -> None:
        # Default behavior: add nothing, just pass the job inward.
        self._wrapped_order.execute()

### Step 4: the concrete decorators (the actual layers)

Each one inherits from `OrderDecorator`, so it automatically gets `self._wrapped_order`. It only has to say **what extra thing it does, and when**:

- `LoggingDecorator` does something **before AND after** the inner work.
- `RiskCheckDecorator` does something **before** the inner work.

The line `self._wrapped_order.execute()` is the "your turn" moment: it hands the job to the next layer inward.

We don't need to write `__init__` in these classes: Python automatically uses the parent's `__init__` from `OrderDecorator`.

In [6]:
class LoggingDecorator(OrderDecorator):
    def execute(self) -> None:
        print("[LOG] Before execution")
        self._wrapped_order.execute()      # hand the job to the inner layer
        print("[LOG] After execution")


class RiskCheckDecorator(OrderDecorator):
    def execute(self) -> None:
        print("[RISK CHECK] Validating order")
        self._wrapped_order.execute()      # hand the job to the inner layer

### Step 5: use it

Read the wrapping line **from the inside out**, exactly like wrapping a gift:

```python
both = LoggingDecorator(RiskCheckDecorator(order))
```

1. Start with `order` (the plain `BasicOrder`, the gift).
2. Wrap it in a `RiskCheckDecorator` (first layer of paper).
3. Wrap **that** in a `LoggingDecorator` (second layer of paper).

We stack exactly the layers we need, decided at runtime. No `LoggedRiskCheckedOrder` class required.

In [7]:
order = BasicOrder()

# Stack exactly the layers you need, decided at RUNTIME.
# No "LoggedRiskCheckedOrder" class required.
both = LoggingDecorator(RiskCheckDecorator(order))
both.execute()

[LOG] Before execution
[RISK CHECK] Validating order
Executing order
[LOG] After execution


### What exactly happened? A slow-motion replay

When we call `both.execute()`, this is the order of events:

| Step | Who is running | What it does |
|---|---|---|
| 1 | `LoggingDecorator.execute()` | prints `[LOG] Before execution` |
| 2 | `LoggingDecorator.execute()` | calls `self._wrapped_order.execute()`, and its wrapped order is the `RiskCheckDecorator` |
| 3 | `RiskCheckDecorator.execute()` | prints `[RISK CHECK] Validating order` |
| 4 | `RiskCheckDecorator.execute()` | calls `self._wrapped_order.execute()`, and its wrapped order is the `BasicOrder` |
| 5 | `BasicOrder.execute()` | prints `Executing order`. Nothing inside it, so it is finished |
| 6 | back in `RiskCheckDecorator.execute()` | nothing left to do, so it returns |
| 7 | back in `LoggingDecorator.execute()` | prints `[LOG] After execution` |

The calls go **inward** (steps 1 to 5), then **return back outward** (steps 6 and 7).
That is why `[LOG] After execution` is printed **last**, even though it is written in the outermost layer.

Picture it as a set of Russian nesting dolls: open each doll on the way in, close each doll on the way out.

## 1.7 The order of the layers matters

Same two layers, stacked the other way round: now **Logging** is inside and **Risk Check** is outside.
Compare the output with the previous one: the risk check now happens **before** anything is logged.

In a real system this matters. For example, if the risk check rejects an order, do you want that attempt to appear in the log or not? You choose by choosing the order of the layers.

In [8]:
reversed_layers = RiskCheckDecorator(LoggingDecorator(BasicOrder()))
reversed_layers.execute()

[RISK CHECK] Validating order
[LOG] Before execution
Executing order
[LOG] After execution


## 1.8 Deciding at RUNTIME, for real

Here is the payoff. The function below builds an order from plain `True`/`False` settings, which could come from a config file, a user's account, or the size of the order.

Compare it with the 16-branch `if/elif` ladder from section 1.3. Here there is **one small `if` per feature**, no matter how many combinations exist.

Notice the variable `order` is re-assigned each time: it starts as the plain order, then becomes "the plain order wrapped in risk-check", then "that thing wrapped in logging". Each wrap goes **around** whatever we have so far.

In [9]:
def build_order(needs_logging: bool, needs_risk_check: bool) -> Order:
    order: Order = BasicOrder()
    if needs_risk_check:
        order = RiskCheckDecorator(order)
    if needs_logging:
        order = LoggingDecorator(order)
    return order


for needs_logging in (False, True):
    for needs_risk_check in (False, True):
        print(f"--- logging={needs_logging}, risk_check={needs_risk_check} ---")
        build_order(needs_logging, needs_risk_check).execute()

--- logging=False, risk_check=False ---
Executing order
--- logging=False, risk_check=True ---
[RISK CHECK] Validating order
Executing order
--- logging=True, risk_check=False ---
[LOG] Before execution
Executing order
[LOG] After execution
--- logging=True, risk_check=True ---
[LOG] Before execution
[RISK CHECK] Validating order
Executing order
[LOG] After execution


## 1.9 Adding a new feature costs ONE class

The business now wants a **notification** sent after every order.

- Naive approach: double the number of classes (4 becomes 8), and copy-paste the notification code into 4 of them.
- Decorator approach: write **one** new small class. Nothing that already exists has to change: not `BasicOrder`, not `LoggingDecorator`, not `RiskCheckDecorator`.

This is called the **Open/Closed Principle**: code should be *open for extension* (you can add new behavior) but *closed for modification* (you don't have to edit code that already works).

In [10]:
class NotificationDecorator(OrderDecorator):
    def execute(self) -> None:
        self._wrapped_order.execute()      # do the inner work FIRST...
        print("[NOTIFY] Order filled: sending email to the customer")  # ...then notify


everything = NotificationDecorator(LoggingDecorator(RiskCheckDecorator(BasicOrder())))
everything.execute()

[LOG] Before execution
[RISK CHECK] Validating order
Executing order
[LOG] After execution
[NOTIFY] Order filled: sending email to the customer


## 1.10 The vocabulary (what to call each piece)

| Official pattern name | In our example | What it means in plain English |
|---|---|---|
| **Component** | `Order` | the shared interface: the "shape" everyone has |
| **Concrete Component** | `BasicOrder` | the real object doing the core job: the gift |
| **Decorator** (base) | `OrderDecorator` | IS a Component AND HAS a Component: the base for wrapping paper |
| **Concrete Decorators** | `LoggingDecorator`, `RiskCheckDecorator`, `NotificationDecorator` | each layer of wrapping paper; each adds one feature |

**In one sentence:** Decorator lets you add behavior to an object at runtime by wrapping it in layers that have the same interface, where each layer does its own bit and then delegates inward, instead of writing a subclass for every combination.

## 1.11 Don't confuse this with Python's `@decorator` syntax

Python has a built-in feature *also* called a "decorator": the `@something` line above a function. It is the **same idea applied to functions** instead of objects: wrap something, do extra work, then call the original.

The cell below adds logging to a plain function. Notice the shape is identical to `LoggingDecorator`: do something before, call the wrapped thing, do something after.

- **Decorator pattern** (Part 1 above): wraps **objects**, uses classes and a shared interface, and the layers are chosen while the program runs.
- **Python `@decorator`**: wraps **functions**, and is usually applied once, where the function is defined.

In [11]:
import functools


def log_calls(func):
    @functools.wraps(func)  # keeps the original function's name and docstring
    def wrapper(*args, **kwargs):
        print(f"[LOG] Before {func.__name__}")
        result = func(*args, **kwargs)   # call the original function
        print(f"[LOG] After {func.__name__}")
        return result
    return wrapper


@log_calls  # exactly the same as writing: place_trade = log_calls(place_trade)
def place_trade(symbol: str, quantity: int) -> None:
    print(f"Buying {quantity} shares of {symbol}")


place_trade("AAPL", 10)

[LOG] Before place_trade
Buying 10 shares of AAPL
[LOG] After place_trade


## 1.12 Common mistakes and quick checklist

**Common mistakes**

1. **Forgetting to call `self._wrapped_order.execute()`.** The layer then "swallows" the call and the real order is never executed. (Sometimes that is intended, e.g. a risk check that *rejects* a bad order. If so, make it obvious in the code.)
2. **Typing the wrapped object as `BasicOrder` instead of `Order`.** Then a decorator can't wrap another decorator, and stacking breaks.
3. **Making a decorator that does NOT inherit from `Order`.** Then it can't be passed to code that expects an `Order`, and it can't be wrapped by another layer.
4. **Not thinking about layer order.** `Logging(RiskCheck(order))` and `RiskCheck(Logging(order))` behave differently (section 1.7).

**Use Decorator when...**

- you have a core object and several **optional, independent extras**, and
- you want to mix and match those extras **per object, at runtime**, and
- you don't want the core class to know about the extras.

---

# Part 2: Adapter

## 2.1 What problem are we solving?

**Goal:** make an incompatible, pre-existing interface work with your code, **without modifying the original class**.

### The example: taking payments

Your shop's checkout code is already written and tested. It works with **any** object that follows your own `PaymentProcessor` interface, which has one method:

```python
pay(amount, card_token) -> True / False
```

- `amount` is in **dollars**, like `49.99`.
- It returns `True` if the payment worked and `False` if it failed.

Now the business wants to use **Stripe**. Stripe gives you their own library (an "SDK", i.e. a "Software Development Kit": code someone else wrote for you to use). But Stripe's SDK speaks a **different language** from your checkout:

| | What YOUR checkout expects | What STRIPE's SDK provides |
|---|---|---|
| Method name | `pay(...)` | `create_charge(...)` |
| Money | dollars as a decimal: `49.99` | whole **cents** as an integer: `4999` |
| Extra info | none | also needs a `currency`, like `"usd"` |
| Result | `True` or `False` | a dictionary like `{"status": "succeeded", ...}` |

And there are **two things you are not allowed to do**:

- You **can't edit Stripe's SDK**: it isn't your code, and your changes would be wiped out the next time you update it.
- You **don't want to edit your checkout**: it already works, and it should not become Stripe-specific. (What if you switch to PayPal next year?)

The solution: put a small **translator** object in the middle. That translator is the **Adapter**.

> Note: the slides only showed the end of this example (`StripeSDK`, `StripeAdapter`, `checkout(adapter, 49.99, "tok_123")` and the vocabulary).
> The code below is a complete Python version built around those same names.

## 2.2 Step 1: the interface YOUR system expects (the "Target")

This is your own rulebook. Your checkout only knows how to talk to a `PaymentProcessor`.

In [12]:
from abc import ABC, abstractmethod


class PaymentProcessor(ABC):
    @abstractmethod
    def pay(self, amount: float, card_token: str) -> bool:
        """Charge `amount` dollars to the card. Return True if it worked."""
        ...


def checkout(processor: PaymentProcessor, amount: float, card_token: str) -> None:
    """Your existing, already-tested checkout code. It knows NOTHING about Stripe."""
    print(f"Checkout: charging ${amount:.2f}")
    if processor.pay(amount, card_token):
        print("Checkout: payment succeeded, order confirmed")
    else:
        print("Checkout: payment FAILED, please try another card")

## 2.3 Step 2: the existing, incompatible class (the "Adaptee")

Pretend this class came from Stripe. **We did not write it and we are not allowed to change it.**
It works perfectly well; it just speaks a different "language" from our checkout.

(In real life you would `import stripe` instead. This fake version just prints what it would do, so we can run it safely.)

In [13]:
class StripeSDK:
    """Pretend this is Stripe's own library. We CANNOT change this code."""

    def __init__(self, api_key: str) -> None:
        self.api_key = api_key

    def create_charge(self, amount_in_cents: int, currency: str, source: str) -> dict:
        print(f"  [StripeSDK] charging {amount_in_cents} cents ({currency}) to card {source}")
        return {"id": "ch_001", "status": "succeeded", "amount": amount_in_cents}

## 2.4 Why can't we just pass Stripe straight into checkout?

Because `checkout` calls `processor.pay(...)`, and `StripeSDK` **has no method called `pay`**. It has `create_charge` instead.
Python finds out when it tries to call `.pay`, and crashes with an `AttributeError` ("this object has no attribute called `pay`").

This is what "incompatible interface" means: the object can do the job, but it doesn't have the shape our code expects.

In [14]:
stripe = StripeSDK(api_key="sk_test_123")

try:
    checkout(stripe, 49.99, "tok_123")
except AttributeError as error:
    print("Crash:", error)

Checkout: charging $49.99
Crash: 'StripeSDK' object has no attribute 'pay'


## 2.5 Step 3: the translator (the "Adapter")

The adapter:

1. **Inherits from `PaymentProcessor`**, so it looks exactly like what our checkout expects.
2. **Holds the Stripe object inside it** (`self._stripe`), just as a decorator holds the thing it wraps.
3. In its `pay()` method, it **translates** every difference from the table in section 2.1:
   - **Method name:** our `pay(...)` becomes Stripe's `create_charge(...)`.
   - **Units:** dollars become cents (`49.99` becomes `4999`).
   - **Missing info:** it fills in `currency="usd"`, which our system never had to think about.
   - **Result:** Stripe's dictionary becomes the plain `True`/`False` our checkout wants.

In [15]:
class StripeAdapter(PaymentProcessor):
    """Looks like a PaymentProcessor on the outside; talks to Stripe on the inside."""

    def __init__(self, stripe: StripeSDK) -> None:
        self._stripe = stripe  # the incompatible object we are wrapping

    def pay(self, amount: float, card_token: str) -> bool:
        # Translation 1: dollars (float) -> cents (int). round(), NOT int(): see the next cell.
        amount_in_cents = round(amount * 100)

        # Translation 2: our method name and arguments -> Stripe's method name and arguments.
        result = self._stripe.create_charge(
            amount_in_cents=amount_in_cents,
            currency="usd",
            source=card_token,
        )

        # Translation 3: Stripe's dictionary -> our simple True/False.
        return result["status"] == "succeeded"

### Why `round()` and not `int()` when converting to cents?

Computers store decimal numbers like `19.99` in binary, and most of them can't be stored *exactly*.
So `19.99 * 100` is not quite `1999`. It comes out a tiny bit **under** it.

- `int()` simply **chops off** everything after the decimal point, so you get `1998`. **The customer is charged one cent too little!**
- `round()` rounds to the **nearest** whole number, so you get `1999`. Correct.

Translating units is exactly the kind of detail an adapter exists to get right, in one place, so the rest of your code never has to worry about it. (For serious money code, Python's `decimal.Decimal` type avoids this problem entirely.)

In [16]:
print("19.99 * 100        =", 19.99 * 100)
print("int(19.99 * 100)   =", int(19.99 * 100), "  <- WRONG, one cent short")
print("round(19.99 * 100) =", round(19.99 * 100), "  <- correct")

19.99 * 100        = 1998.9999999999998
int(19.99 * 100)   = 1998   <- WRONG, one cent short
round(19.99 * 100) = 1999   <- correct


## 2.6 Step 4: use it

We wrap the Stripe object in the adapter, and hand the **adapter** to our checkout.
Neither `checkout` nor `StripeSDK` was changed by a single line.

In [17]:
stripe = StripeSDK(api_key="sk_test_123")
adapter = StripeAdapter(stripe)
checkout(adapter, 49.99, "tok_123")

Checkout: charging $49.99
  [StripeSDK] charging 4999 cents (usd) to card tok_123
Checkout: payment succeeded, order confirmed


### Slow-motion replay

| Step | Who is running | What it does |
|---|---|---|
| 1 | `checkout(...)` | prints `Checkout: charging $49.99`, then calls `processor.pay(49.99, "tok_123")`. It thinks it is talking to an ordinary `PaymentProcessor` |
| 2 | `StripeAdapter.pay(...)` | converts `49.99` dollars into `4999` cents |
| 3 | `StripeAdapter.pay(...)` | calls Stripe's `create_charge(4999, "usd", "tok_123")` |
| 4 | `StripeSDK.create_charge(...)` | does the charge and returns `{"status": "succeeded", ...}` |
| 5 | `StripeAdapter.pay(...)` | turns that dictionary into `True` and returns it |
| 6 | `checkout(...)` | receives `True` and prints `payment succeeded, order confirmed` |

`checkout` never learned that Stripe exists. `StripeSDK` never learned that `checkout` exists. Only the adapter knows about both.

## 2.7 The payoff: switching providers costs ONE new adapter

Next year the business also wants PayPal. PayPal's SDK speaks *yet another* language: a different method name, money as a **text string** like `"49.99"`, and a result of `"APPROVED"`/`"DECLINED"` instead of a dictionary.

We write **one more adapter**. `checkout` stays exactly the same.

In [18]:
class PayPalSDK:
    """Pretend this is PayPal's library. We CANNOT change this code either."""

    def make_payment(self, total: str, payment_token: str) -> str:
        print(f"  [PayPalSDK] paying {total} with token {payment_token}")
        return "APPROVED"


class PayPalAdapter(PaymentProcessor):
    def __init__(self, paypal: PayPalSDK) -> None:
        self._paypal = paypal

    def pay(self, amount: float, card_token: str) -> bool:
        total = f"{amount:.2f}"  # 49.99 (number) -> "49.99" (text, always 2 decimals)
        return self._paypal.make_payment(total, card_token) == "APPROVED"


checkout(PayPalAdapter(PayPalSDK()), 49.99, "tok_456")

Checkout: charging $49.99
  [PayPalSDK] paying 49.99 with token tok_456
Checkout: payment succeeded, order confirmed


## 2.8 The vocabulary (what to call each piece)

| Official pattern name | In our example | What it means in plain English |
|---|---|---|
| **Target** | `PaymentProcessor` | the interface **your system expects** |
| **Adaptee** | `StripeSDK` | the **existing, incompatible** thing being wrapped |
| **Adapter** | `StripeAdapter` | **translates** between the two |
| **Client** | `checkout(...)` | the code that uses the Target and never knows an adapter is involved |

**In one sentence:** Adapter lets an object with an incompatible interface be used wherever the expected interface is required, by wrapping and translating, without modifying the original class.

## 2.9 Decorator vs Adapter: the side-by-side cheat sheet

Both wrap an object and both pass calls on to it. Here is how to tell them apart:

| Question | Decorator | Adapter |
|---|---|---|
| **Why** are you wrapping? | to **ADD** behavior | to **TRANSLATE** an interface |
| How many interfaces are involved? | **ONE**: the wrapper and the wrapped object have the **same** interface (`Order` wraps `Order`) | **TWO**: the wrapper has the **Target** interface, the wrapped object has a **different** one (`PaymentProcessor` wraps `StripeSDK`) |
| Can you stack many of them? | Yes, that's the whole point (`Logging(RiskCheck(order))`) | Normally just one, between your code and the foreign code |
| Does the result do *more* than the original? | Yes: same job, plus extras | No: same job, just reachable through a different "plug" |
| Analogy | gift wrapping | travel plug adapter |

**Quick test:** look at the method the wrapper exposes and the method it calls inside.
- Same name (`execute()` calls `execute()`): **Decorator**.
- Different name (`pay()` calls `create_charge()`): **Adapter**.